# RNN 數學推導總整理

RNN（Recurrent Neural Network）是一種可以處理「序列資料」的神經網路，將時間展開的 RNN 視為深層網路，並透過時間反向傳播梯度，例如：

* 文字（句子）
* 時間序列（股價）
* 語音

其核心思想是：每一步都會保留「前一刻的記憶」，每一步都依賴前一步（時間關聯）。

<img src="image/rnn.png" alt="rnn" style="width: 80%;"/>

參考圖片標示與以下不同，但概念是相同的。

## 向前傳播

給定：

$$
a_t = W_x x_t + W_h h_{t-1} + b
$$

$$
h_t = \phi(a_t)
$$

$$
o_t = W_y h_t + c
$$

$$
\hat{y}_t = \text{softmax}(o_t)
$$

總損失：

$$
L = \sum_{t=1}^{T} L_t
$$

（分類問題）

$$
L_t = -\sum_i y_{t,i} \log \hat{y}_{t,i}
$$

## 向後傳播

### 輸出層梯度

對於 softmax + cross entropy：

$$
\delta_t^o = \frac{\partial L_t}{\partial o_t} = \hat{y}_t - y_t
$$

再者：

$$
h_t = \frac{\partial o_t}{\partial W_y}
$$

$$
\frac{\partial L_t}{\partial W_y} =\frac{\partial L_t}{\partial o_t}\frac{\partial o_t}{\partial W_y} = \delta_t^o h_t 
$$

因此：

$$
\boxed{
\frac{\partial L}{\partial W_y}=\sum_{t=1}^{T} \delta_t^o h_t^\top
}
$$

$$
\boxed{
\frac{\partial L}{\partial c}=\sum_{t=1}^{T} \delta_t^o
}
$$

### δ 的定義與推導

#### RNN 在時間步 t 同時受到當前與未來時間的影響

對於基本 RNN，在時間步 $t$ 的計算流程為：

$$
a_t \rightarrow h_t \rightarrow o_t \rightarrow L_t
$$

同時，$h_t$ 還會透過 recurrent connection 影響未來時間步：

$$
a_t \rightarrow h_t \rightarrow a_{t+1} \rightarrow \cdots \rightarrow L_{t+1}, L_{t+2}, \dots
$$

其中：

- $a_t$ 是 hidden layer 在 activation 前的線性組合
- 所有參數 $W_x, W_h, b$ 都先影響 $a_t$
- 因此將誤差統一定義在 $a_t$ 上，能讓參數梯度寫成一致且簡潔的矩陣形式

---

#### 定義誤差項

定義：

$$
\delta_t^o := \frac{\partial L}{\partial o_t}
\qquad
\delta_t^h := \frac{\partial L}{\partial h_t}
\qquad
\delta_t^a := \frac{\partial L}{\partial a_t}
$$

其中最重要的是 hidden layer pre-activation 的誤差：

$$
\delta_t^a = \frac{\partial L}{\partial a_t}
$$

---

#### 由鏈式法則推導

因為

$$
h_t = \phi(a_t)
$$

所以

$$
\delta_t^a=
\frac{\partial L}{\partial a_t}=
\frac{\partial L}{\partial h_t}
\odot
\frac{\partial h_t}{\partial a_t}=
\delta_t^h \odot \phi'(a_t)
$$

---

#### 隱藏記憶對 loss 的兩條影響路徑

$h_t$ 影響 loss 有兩條路徑：

1. 路徑 A：影響當前輸出

$$
h_t \rightarrow o_t \rightarrow L_t
$$

由於

$$
o_t = W_y h_t + c
$$

因此：

$$
\delta_t^h\Big|_{\text{current}}=
\frac{\partial L}{\partial h_t}\Big|_{\text{current}}=
\frac{\partial L}{\partial o_t} \cdot \frac{\partial o_t}{\partial h_t}=
W_y^\top \delta_t^o
$$

---

2. 路徑 B：影響未來時間

$$
h_t \rightarrow a_{t+1} \rightarrow \cdots \rightarrow L
$$

因為

$$
a_{t+1} = W_h h_t + W_x x_{t+1} + b
$$

所以：

$$
\delta_t^h\Big|_{\text{future}}=
\frac{\partial L}{\partial h_t}\Big|_{\text{future}}=
\frac{\partial L}{\partial a_{t+1}} \cdot \frac{\partial a_{t+1}}{\partial h_t}=
W_h^\top \delta_{t+1}^a
$$

---

#### 合併兩條路徑

因此：

$$
\delta_t^h=
\frac{\partial L}{\partial h_t}=
\delta_t^h\Big|_{\text{current}} + \delta_t^h\Big|_{\text{future}}=
W_y^\top \delta_t^o + W_h^\top \delta_{t+1}^a
$$

再乘上啟動函數的導數，可得：

$$
\delta_t^a=
\delta_t^h \odot \phi'(a_t)
$$

$$
\boxed{
\delta_t^a=
\left(
W_y^\top \delta_t^o + W_h^\top \delta_{t+1}^a
\right)\odot \phi'(a_t)
}
$$

這就是 RNN 在 BPTT 中的核心遞迴公式。

---

若啟動函數為tanh

若

$$
h_t = \tanh(a_t)
$$

則

$$
\phi'(a_t)=1-h_t^2
$$

因此：

$$
\boxed{
\delta_t^a=
\left(
W_y^\top \delta_t^o + W_h^\top \delta_{t+1}^a
\right)\odot (1-h_t^2)
}
$$

---

#### 參數梯度

由於

$$
a_t = W_x x_t + W_h h_{t-1} + b
$$

所以可直接得到各參數的梯度。

1. 對 $W_h$

$$
\boxed{
\frac{\partial L}{\partial W_h}=
\sum_{t=1}^{T}
\delta_t^a \, h_{t-1}^\top
}
$$

2. 對 $W_x$

$$
\boxed{
\frac{\partial L}{\partial W_x}=
\sum_{t=1}^{T}
\delta_t^a \, x_t^\top
}
$$

3. 對 bias $b$

$$
\boxed{
\frac{\partial L}{\partial b}=
\sum_{t=1}^{T}
\delta_t^a
}
$$

---

補充：邊界條件

若 loss 只定義到時間步 $T$，則在反向傳播時通常設：

$$
\delta_{T+1}^a = 0
$$

因此最後一步變成：

$$
\delta_T^a=
\left(W_y^\top \delta_T^o\right)\odot \phi'(a_T)
$$

## RNN 的梯度消失與爆炸問題

### 梯度消失

$$
\delta_t^h \sim (W_h^T)^k
$$

若 $\|W_h\| < 1$：

→ 梯度指數衰減

### 梯度爆炸

若 $\|W_h\| > 1$：

→ 梯度指數爆炸

## Truncated BPTT

實務中限制回傳步數 $K$：

$$
\delta_t^h \approx 0 \quad \text{for } t < T-K
$$

優點：
- 計算效率高
- 訓練穩定

缺點：
- 長期依賴學習能力下降